# California Housing Dataset: End-to-End Data Science Pipeline

**Dataset:** California Housing (1990 US Census)  
**Scope:** Exploratory Data Analysis · Preprocessing · PCA · Machine Learning  
**Models compared:** Linear Regression · Random Forest · XGBoost

---

This notebook walks through a complete Data Science pipeline applied to real estate price prediction in California. Each section corresponds to a logical stage of the workflow, from raw data loading to final model evaluation.

## Table of Contents

1. [Library Imports](#section-1)
2. [Data Loading](#section-2)
3. [Exploratory Data Analysis (EDA)](#section-3)
4. [Advanced Visualization](#section-4)
5. [Preprocessing](#section-5)
6. [PCA: Dimensionality Reduction](#section-6)
7. [Correlation Circle](#section-7)
8. [Machine Learning](#section-8)

---
## 1 · Library Imports

All dependencies are imported upfront. The stack covers data manipulation (`pandas`, `polars`, `numpy`), visualization (`matplotlib`, `seaborn`), and machine learning via Scikit-learn and XGBoost.

In [ ]:
import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler
from sklearn.decomposition  import PCA

from sklearn.linear_model   import LinearRegression
from sklearn.ensemble       import RandomForestRegressor
from xgboost                import XGBRegressor

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

---
## 2 · Data Loading

The dataset is loaded from a local CSV file. `head(10)` gives a first look at the structure: each row is a geographic district in California, with numeric features and one text column (`ocean_proximity`).

In [ ]:
df = pd.read_csv("../data/housing.csv")

# First look at the data
df.head(10)

---
## 3 · Exploratory Data Analysis (EDA)

Before any modelling, the data must be understood. This cell covers four essential inspection steps: column types, summary statistics, missing value counts, and the distribution of the categorical variable.

Key finding: `total_bedrooms` contains **207 missing values** out of 20,640 rows, a real issue to address in preprocessing. The target variable `median_house_value` is artificially capped at $500,001.

In [ ]:
# Column types and general information
df.info()

# Descriptive statistics
df.describe()

# Missing values
df.isnull().sum()

# Distribution of the categorical variable
df['ocean_proximity'].value_counts()

---
## 4 · Advanced Visualization

Three complementary visualizations are produced here.

- **Histograms** reveal skewed distributions and the visible cap on `median_house_value`.
- **Correlation heatmap** identifies `median_income` as the strongest predictor (r approx. 0.69).
- **Geographic scatter plot** reconstructs the map of California using GPS coordinates, coloring by price and sizing by population density.

In [ ]:
df.hist(figsize=(14, 10), bins=50, edgecolor='black')
plt.suptitle("Distribution of all variables", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(
    df.corr(numeric_only=True),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)
plt.title("Correlation matrix", fontsize=14)
plt.show()

In [ ]:
df.plot(
    kind="scatter",
    x="longitude",
    y="latitude",
    alpha=0.3,
    figsize=(12, 8),
    c="median_house_value",
    cmap="hot_r",
    colorbar=True,
    s=df["population"] / 100
)
plt.title("Housing prices map - California")
plt.show()

---
## 5 · Preprocessing

Four operations prepare the data for modelling:

1. **Missing values** in `total_bedrooms` are filled with the column median, which is robust against outliers.
2. **One-hot encoding** converts `ocean_proximity` into binary columns; ML algorithms cannot handle raw strings.
3. **Feature / target split** separates `X` (inputs) from `y` (`median_house_value`).
4. **Standardization** (zero mean, unit variance) prevents high-magnitude variables from dominating and is a prerequisite for PCA.

In [ ]:
# Handle missing values
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)

# Encode the categorical variable
df = pd.get_dummies(df, columns=['ocean_proximity'])

# Split features and target
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

---
## 6 · PCA: Dimensionality Reduction

Principal Component Analysis projects the ~13-dimensional feature space into 2 dimensions while retaining the maximum variance. `explained_variance_ratio_` reports what percentage of total information each component captures. The scatter plot shows how the 20,640 districts distribute in this reduced space.

In [ ]:
# Build the PCA model with 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Explained variance
print("Variance explained per component:", pca.explained_variance_ratio_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

# Plot the PCA projection
plt.figure(figsize=(10, 7))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    alpha=0.3,
    s=5,
    c="steelblue"
)
plt.xlabel("PC1 (first principal component)")
plt.ylabel("PC2 (second principal component)")
plt.title("PCA projection of housing data")
plt.show()

---
## 7 · Correlation Circle

The correlation circle maps each original variable as an arrow in the PCA plane. **Arrow length** indicates how well the variable is represented by the two components. **Arrow direction** encodes correlations: arrows pointing the same way are positively correlated; opposing arrows indicate negative correlation; perpendicular arrows are uncorrelated.

Expected pattern: `total_rooms`, `total_bedrooms`, `population`, and `households` cluster together, while `median_income` points in a distinct direction, confirming its relative independence and predictive strength.

In [ ]:
pcs = pca.components_

plt.figure(figsize=(10, 10))

for i, feature in enumerate(X.columns):
    plt.arrow(
        0, 0,
        pcs[0, i],
        pcs[1, i],
        head_width=0.03,
        head_length=0.02,
        fc='steelblue',
        ec='steelblue'
    )
    plt.text(
        pcs[0, i] * 1.12,
        pcs[1, i] * 1.12,
        feature,
        fontsize=9
    )

circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='--')
plt.gca().add_artist(circle)

plt.xlim(-1.2, 1.2)
plt.ylim(-1.2, 1.2)
plt.axhline(0, color='grey', linewidth=0.8)
plt.axvline(0, color='grey', linewidth=0.8)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Correlation circle - PCA")
plt.show()

---
## 8 · Machine Learning

The pipeline concludes with a standard train/test split (80 / 20) followed by three regression models of increasing complexity.

- **Linear Regression:** the baseline; assumes a linear relationship between features and price.
- **Random Forest:** an ensemble of 100 parallel decision trees whose predictions are averaged.
- **XGBoost:** gradient boosting; trees are trained sequentially, each correcting the residual errors of the previous one.

Performance is measured with **R²** (proportion of variance explained, 0 to 1) and **RMSE / MAE** in dollars.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

# XGBoost
xgb = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

In [ ]:
def evaluate(y_true, y_pred, model_name):
    print(f"=== {model_name} ===")
    print(f"  R2   : {r2_score(y_true, y_pred):.4f}")
    print(f"  RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):,.0f} $")
    print(f"  MAE  : {mean_absolute_error(y_true, y_pred):,.0f} $")
    print()

evaluate(y_test, pred_lr,  "Linear Regression")
evaluate(y_test, pred_rf,  "Random Forest")
evaluate(y_test, pred_xgb, "XGBoost")

---
## Conclusion

This notebook demonstrated a complete Data Science pipeline on the California Housing Dataset:

- Missing values handled with median imputation
- Categorical variable encoded via one-hot encoding
- Data standardized for both PCA and distance-based models
- Dimensionality reduction applied with PCA and visualized through the correlation circle
- Three regression models trained and evaluated with R2, RMSE, and MAE

The most important takeaway: **median income** is by far the strongest predictor of housing prices in California, a real insight and not an artificial one.

Possible next steps: hyperparameter tuning with `GridSearchCV`, feature engineering (rooms-per-household, bedrooms-per-room), or testing LightGBM and CatBoost.